# Execução local — Dashboard SISCAN

Inicia a aplicação Streamlit no compute do Databricks para desenvolvimento e testes. Não modifica dados das camadas Medallion nem o deploy público.

Execute a partir da pasta `notebooks/` do repositório Git, com diretório de trabalho correspondente (Databricks Runtime 14+ em compute clássico). Para consultas à Gold, configure os segredos do Streamlit ou as variáveis de ambiente documentadas em `src/app/README.md`. Não insira credenciais neste notebook.

Use compute dedicado e acesso autenticado. O servidor permanece ativo enquanto o processo e o compute estiverem disponíveis. O encaminhamento de porta depende das permissões do workspace. Em serverless, o diretório de trabalho e o proxy podem variar; utilize um compute clássico dedicado para este fluxo ou configure o encaminhamento autorizado.

In [0]:
%pip install -r ../src/app/requirements.txt

In [0]:
import os
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

PORT = 8501
STARTUP_TIMEOUT = 60
STOP_TIMEOUT = 10
HOST = "0.0.0.0"


def localizar_app() -> Path:
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root / "src" / "app"
        if (candidate / "main.py").is_file() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Execute o notebook a partir do repositório siscan-data-analytics.")


APP_DIR = localizar_app()
APP_PATH = APP_DIR / "main.py"
HEALTH_URL = f"http://127.0.0.1:{PORT}/_stcore/health"


def porta_disponivel(porta: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind((HOST, porta))
        except OSError:
            return False
        return True


def servidor_pronto() -> bool:
    try:
        with urlopen(HEALTH_URL, timeout=2) as response:
            return response.status == 200 and response.read().strip() == b"ok"
    except (URLError, TimeoutError, OSError):
        return False


def parar_streamlit() -> None:
    processo = globals().get("streamlit_process")
    if processo is None or processo.poll() is not None:
        return
    processo.terminate()
    try:
        processo.wait(timeout=STOP_TIMEOUT)
    except subprocess.TimeoutExpired:
        processo.kill()
        processo.wait(timeout=STOP_TIMEOUT)


def aguardar_inicializacao(processo: subprocess.Popen) -> None:
    deadline = time.monotonic() + STARTUP_TIMEOUT
    while time.monotonic() < deadline:
        if processo.poll() is not None:
            raise RuntimeError(f"Streamlit encerrou com código {processo.returncode}.")
        if servidor_pronto():
            return
        time.sleep(0.5)
    raise TimeoutError(f"Streamlit não respondeu em {STARTUP_TIMEOUT} segundos.")


def iniciar_streamlit() -> tuple[subprocess.Popen, Path]:
    # Não interrompe processos de outros notebooks ou serviços.
    if not porta_disponivel(PORT):
        raise RuntimeError(f"Porta {PORT} ocupada. Encerre apenas o processo responsável ou altere PORT.")

    command = [
        sys.executable, "-m", "streamlit", "run", str(APP_PATH),
        "--server.port", str(PORT),
        "--server.address", HOST,
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
    ]
    log_dir = Path(tempfile.mkdtemp(prefix="siscan_streamlit_"))
    log_path = log_dir / "streamlit.log"

    with log_path.open("w", encoding="utf-8") as log:
        processo = subprocess.Popen(
            command,
            cwd=str(APP_DIR),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    return processo, log_path

In [0]:
parar_streamlit()
streamlit_process, streamlit_log = iniciar_streamlit()

try:
    aguardar_inicializacao(streamlit_process)
except (RuntimeError, TimeoutError) as error:
    parar_streamlit()
    raise RuntimeError(f"{error} Consulte o log local: {streamlit_log}") from error

print(f"Streamlit disponível no compute | porta {PORT} | PID {streamlit_process.pid}")
print(f"Log local: {streamlit_log}")

In [0]:
from html import escape
from urllib.parse import urlsplit

from IPython.display import HTML, display


def url_proxy_classico() -> str | None:
    def config(key: str) -> str:
        try:
            return spark.conf.get(key, "") or ""
        except Exception:
            return ""

    host = config("spark.databricks.workspaceUrl")
    workspace = config("spark.databricks.clusterUsageTags.clusterOwnerOrgId")
    cluster = config("spark.databricks.clusterUsageTags.clusterId")
    if not host or not workspace or not cluster:
        return None

    base = host if host.startswith("https://") else f"https://{host}"
    return f"{base.rstrip('/')}/driver-proxy/o/{workspace}/{cluster}/{PORT}/"


def publicar_link() -> None:
    # localhost no navegador não é o localhost do compute remoto.
    url = os.getenv("SISCAN_STREAMLIT_URL") or url_proxy_classico()
    if not url:
        print(f"Servidor pronto no compute. Encaminhe a porta {PORT} pelo recurso autorizado do Databricks.")
        print("Opcionalmente, configure SISCAN_STREAMLIT_URL com a URL autenticada e execute esta célula novamente.")
        return

    parsed = urlsplit(url)
    if parsed.scheme != "https" or not parsed.netloc or parsed.username or parsed.password:
        raise ValueError("Use uma URL HTTPS autenticada, sem credenciais embutidas.")
    html = (
        f'<a href="{escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">'
        'Abrir dashboard SISCAN</a>'
    )
    if "displayHTML" in globals():
        displayHTML(html)
    else:
        display(HTML(html))


if streamlit_process.poll() is not None or not servidor_pronto():
    raise RuntimeError("Servidor indisponível. Execute a célula de inicialização.")

publicar_link()

## Encerrar o serviço

Para liberar a porta e finalizar apenas a instância iniciada por este notebook, execute manualmente `parar_streamlit()` em uma nova célula. A reexecução da inicialização faz o encerramento controlado da instância anterior, desde que permaneça na mesma sessão Python. Se uma sessão for reiniciada, uma porta ocupada gera erro em vez de interromper processos desconhecidos.

A verificação HTTP confirma a disponibilidade do servidor, não a conexão do dashboard ao SQL Warehouse. Verifique as consultas na interface da aplicação.

Se a instalação de dependências reiniciar a sessão Python, execute novamente as células de configuração e inicialização. Para reiniciar somente o servidor, execute a célula de inicialização sem repetir a instalação.